# SentinelX: A Privacy-Preserving Cyber Fraud Detection and Digital Evidence Integrity Platform Using Threat Intelligence

**Author:** Manvith  
**Program:** IBM SkillsBuild Cyber Security BootCamp 2026 — BharatCares  
**Submission File:** `Manvith_SentinelX.ipynb`  
**Date:** 2026

---

## Abstract

SentinelX is a privacy-preserving, heuristic-based cyber fraud detection prototype built entirely in Python. It accepts suspicious URLs and message text, evaluates them against a configurable set of phishing and fraud indicators, assigns a risk classification (Low / Medium / High), verifies the SHA-256 integrity of local digital evidence files, and generates structured incident reports. All analysis runs locally without sending data to external services and without storing personally identifiable information. Synthetic examples are used throughout to ensure the notebook runs offline in any Python 3.8+ environment.

---

## 1. Problem Statement

Cyber fraud — phishing URLs, smishing (SMS phishing), vishing (voice phishing), and social-engineering attacks — causes billions of dollars of financial and reputational damage every year. Victims often have no quick, free, offline tool that can help them:

* Evaluate whether a URL looks suspicious before clicking it.
* Assess whether an SMS or email message contains common fraud language.
* Preserve and verify the integrity of digital evidence during an incident.
* Generate a structured, exportable incident report.

Most commercial threat-intelligence platforms require paid subscriptions or API keys, making them inaccessible to students, small organisations, and first-responders. SentinelX addresses this gap with a fully offline, open-source prototype.

---

## 2. Objectives

1. Implement heuristic URL analysis that detects suspicious structural patterns without visiting the URL.
2. Implement heuristic message analysis that identifies common phishing language patterns.
3. Provide a transparent, explainable risk score (Low / Medium / High) with reasons.
4. Provide SHA-256 based digital evidence integrity verification.
5. Generate structured incident reports exportable as JSON and plain text.
6. Run entirely offline — no API keys, no external service calls.
7. Validate the implementation with at least 10 meaningful test cases.

---

## 3. Imports & Configuration

In [1]:
# ── Standard-library only ── no external packages required for core logic ──
import re
import hashlib
import json
import os
import datetime
import textwrap
from urllib.parse import urlparse, parse_qs
from pathlib import Path

print("All imports successful.")

All imports successful.


In [2]:
# ── Configuration ─────────────────────────────────────────────────────────

# Risk score thresholds
RISK_LOW_MAX    = 2   # score 0-2  → Low
RISK_MEDIUM_MAX = 5   # score 3-5  → Medium
                      # score 6+   → High

# URL indicator weights (each pattern adds this many points to the score)
URL_WEIGHTS = {
    "ip_address_host":        3,
    "excessive_subdomains":   2,
    "misleading_brand":       2,
    "suspicious_tld":         2,
    "url_shortener":          2,
    "long_url":               1,
    "suspicious_keywords":    2,
    "at_symbol_in_url":       3,
    "double_slash_redirect":  2,
    "hex_or_percent_encoded": 2,
    "non_standard_port":      1,
    "http_not_https":         1,
    "multiple_dots":          1,
    "numeric_subdomain":      1,
}

# Message indicator weights
MSG_WEIGHTS = {
    "urgency_language":       2,
    "prize_or_reward":        2,
    "threat_language":        3,
    "sensitive_data_request": 3,
    "impersonation":          2,
    "suspicious_link":        2,
    "generic_greeting":       1,
    "grammar_errors":         1,
    "call_to_action":         1,
    "otp_or_pin_mention":     2,
}

# Suspicious TLDs commonly abused in phishing
SUSPICIOUS_TLDS = {
    ".tk", ".ml", ".ga", ".cf", ".gq", ".xyz", ".top", ".club",
    ".online", ".site", ".tech", ".loan", ".win", ".click", ".download",
    ".review", ".stream", ".gdn", ".racing", ".accountant"
}

# Well-known URL shorteners
URL_SHORTENERS = {
    "bit.ly", "tinyurl.com", "t.co", "goo.gl", "ow.ly", "is.gd",
    "buff.ly", "adf.ly", "short.io", "rb.gy", "cutt.ly", "tiny.cc"
}

# Brand keywords that may indicate brand impersonation in a URL
BRAND_KEYWORDS = [
    "paypal", "amazon", "apple", "microsoft", "google", "netflix",
    "facebook", "instagram", "twitter", "linkedin", "ebay", "sbi",
    "hdfc", "icici", "axis", "paytm", "phonepe", "bhim", "uidai",
    "irctc", "incometax", "epfo"
]

# Suspicious path/query keywords
SUSPICIOUS_URL_KEYWORDS = [
    "login", "signin", "verify", "secure", "account", "update",
    "confirm", "billing", "payment", "password", "credential",
    "support", "alert", "suspended", "restore", "unlock"
]

# Phishing message patterns
MSG_PATTERNS = {
    "urgency_language": re.compile(
        r"\b(urgent|immediately|act now|expires|limited time|last chance|"
        r"within \d+ hours?|today only|right now|don.t delay|hurry)\b",
        re.IGNORECASE),
    "prize_or_reward": re.compile(
        r"\b(won|winner|prize|reward|gift|free|lottery|lucky draw|"
        r"congratulations|selected|claim|cashback|bonus)\b",
        re.IGNORECASE),
    "threat_language": re.compile(
        r"\b(suspended|blocked|terminated|deactivated|legal action|"
        r"arrested|penalty|fine|court|police|fraud detected|compromised)\b",
        re.IGNORECASE),
    "sensitive_data_request": re.compile(
        r"\b(password|otp|pin|cvv|card number|aadhaar|pan number|"
        r"bank details|account number|social security|ssn|credit card)\b",
        re.IGNORECASE),
    "impersonation": re.compile(
        r"\b(sbi|hdfc|icici|axis bank|paytm|uidai|irctc|income tax|"
        r"epfo|amazon|microsoft|apple|google|paypal|netflix|rbi|npci|"
        r"nsdl|sebi)\b",
        re.IGNORECASE),
    "suspicious_link": re.compile(
        r"(https?://\S+|bit\.ly/\S+|tinyurl\.com/\S+|click here|visit now|"
        r"tap the link|follow the link|open the link)",
        re.IGNORECASE),
    "generic_greeting": re.compile(
        r"\b(dear customer|dear user|dear member|valued customer|"
        r"dear account holder)\b",
        re.IGNORECASE),
    "grammar_errors": re.compile(
        r"(kindly do the needful|do the needful|"
        r"revert back|reply back|\bplz\b)",
        re.IGNORECASE),
    "call_to_action": re.compile(
        r"\b(click|tap|call|reply|respond|submit|update|verify|confirm|download)\b",
        re.IGNORECASE),
    "otp_or_pin_mention": re.compile(
        r"\b(otp|one.time password|pin|mpin|t?pin)\b",
        re.IGNORECASE),
}

print("Configuration loaded.")

Configuration loaded.


---

## 4. URL Analysis Module

In [3]:
def analyze_url(url: str) -> dict:
    """
    Perform heuristic analysis of a URL string.

    Parameters
    ----------
    url : str
        The URL string to analyse. The URL is NEVER visited or fetched.

    Returns
    -------
    dict
        {
          'valid':       bool,
          'url':         str,
          'indicators':  list[dict],   # {name, description, weight}
          'score':       int,
          'risk_level':  str,          # 'Low' | 'Medium' | 'High'
          'summary':     str,
          'disclaimer':  str
        }
    """
    result = {
        "valid": False,
        "url": url,
        "indicators": [],
        "score": 0,
        "risk_level": "Low",
        "summary": "",
        "disclaimer": (
            "DISCLAIMER: This is a heuristic analysis only. A high score does not "
            "confirm that the URL is malicious. A low score does not confirm safety. "
            "Never visit unrecognised URLs."
        )
    }

    # ── Validate input ────────────────────────────────────────────────────
    if not url or not isinstance(url, str):
        result["summary"] = "Empty or invalid input."
        return result

    url = url.strip()
    result["url"] = url

    # Ensure the URL has a scheme for parsing
    if not re.match(r'^https?://', url, re.IGNORECASE):
        if re.match(r'^\w', url):
            url_to_parse = "http://" + url
        else:
            result["summary"] = "URL does not start with a valid scheme."
            return result
    else:
        url_to_parse = url

    try:
        parsed = urlparse(url_to_parse)
    except Exception as exc:
        result["summary"] = f"URL could not be parsed: {exc}"
        return result

    hostname = (parsed.hostname or "").lower()
    path     = parsed.path.lower()
    query    = parsed.query.lower()
    scheme   = parsed.scheme.lower()
    port     = parsed.port
    full_url_lower = url.lower()

    result["valid"] = bool(hostname)
    if not result["valid"]:
        result["summary"] = "Could not extract a hostname from the URL."
        return result

    indicators = []
    score = 0

    def add(name, description):
        w = URL_WEIGHTS.get(name, 1)
        indicators.append({"name": name, "description": description, "weight": w})
        return w

    # ── 1. IP address as host ─────────────────────────────────────────────
    ip_pattern = re.compile(
        r'^(\d{1,3}\.){3}\d{1,3}$'
    )
    if ip_pattern.match(hostname):
        score += add("ip_address_host",
            f"Host '{hostname}' is a raw IP address. Legitimate services "
            f"almost never use raw IPs in URLs sent to users.")

    # ── 2. Excessive subdomains ───────────────────────────────────────────
    subdomain_count = hostname.count(".")
    if subdomain_count >= 3:
        score += add("excessive_subdomains",
            f"Hostname '{hostname}' has {subdomain_count} dots, suggesting "
            f"deeply nested subdomains sometimes used to hide the real domain.")

    # ── 3. Brand keyword in hostname (not as the registered domain) ───────
    for brand in BRAND_KEYWORDS:
        if brand in hostname:
            # Check if it's actually the brand's own domain (simple heuristic)
            parts = hostname.split(".")
            registered = ".".join(parts[-2:]) if len(parts) >= 2 else hostname
            if brand not in registered:
                score += add("misleading_brand",
                    f"Brand keyword '{brand}' appears in the hostname "
                    f"'{hostname}' but not as the registered domain "
                    f"'{registered}', suggesting possible impersonation.")
                break

    # ── 4. Suspicious TLD ─────────────────────────────────────────────────
    for tld in SUSPICIOUS_TLDS:
        if hostname.endswith(tld):
            score += add("suspicious_tld",
                f"TLD '{tld}' is commonly abused in phishing and free "
                f"domain campaigns.")
            break

    # ── 5. URL shortener ──────────────────────────────────────────────────
    for shortener in URL_SHORTENERS:
        if hostname == shortener or hostname.endswith("." + shortener):
            score += add("url_shortener",
                f"'{hostname}' is a known URL-shortening service. Shortened "
                f"URLs conceal the actual destination.")
            break

    # ── 6. Long URL ───────────────────────────────────────────────────────
    if len(url) > 100:
        score += add("long_url",
            f"URL length is {len(url)} characters. Very long URLs sometimes "
            f"embed encoded payloads or obfuscate the destination.")

    # ── 7. Suspicious keywords in path/query ──────────────────────────────
    matched_keywords = [
        kw for kw in SUSPICIOUS_URL_KEYWORDS
        if kw in path or kw in query
    ]
    if matched_keywords:
        score += add("suspicious_keywords",
            f"Suspicious keyword(s) found in path/query: "
            f"{', '.join(matched_keywords)}. These are common in credential-"
            f"harvesting pages.")

    # ── 8. @ symbol in URL (hides real destination) ───────────────────────
    if "@" in url:
        score += add("at_symbol_in_url",
            "The '@' character in a URL causes browsers to ignore everything "
            "before it; it is a classic technique to disguise the real host.")

    # ── 9. Double-slash redirect ──────────────────────────────────────────
    if re.search(r'(?<!:)//', path + query):
        score += add("double_slash_redirect",
            "A double-slash ('//') in the URL path or query may indicate an "
            "open-redirect attempt.")

    # ── 10. Heavy hex / percent encoding ─────────────────────────────────
    pct_count = full_url_lower.count("%")
    if pct_count >= 3:
        score += add("hex_or_percent_encoded",
            f"URL contains {pct_count} percent-encoded characters, which "
            f"can be used to obfuscate malicious content.")

    # ── 11. Non-standard port ─────────────────────────────────────────────
    if port and port not in (80, 443, 8080, 8443):
        score += add("non_standard_port",
            f"Port {port} is non-standard. Phishing pages sometimes run on "
            f"uncommon ports to avoid detection.")

    # ── 12. HTTP (not HTTPS) ──────────────────────────────────────────────
    if scheme == "http":
        score += add("http_not_https",
            "URL uses HTTP instead of HTTPS. While not conclusive, lack of "
            "TLS is a concern for pages handling sensitive data.")

    # ── 13. Multiple consecutive dots ─────────────────────────────────────
    if re.search(r'\.{2,}', hostname):
        score += add("multiple_dots",
            "Multiple consecutive dots in the hostname is unusual and may "
            "indicate an attempt to confuse parsers.")

    # ── 14. Numeric subdomain ─────────────────────────────────────────────
    if re.match(r'^\d+\.', hostname):
        score += add("numeric_subdomain",
            f"Hostname '{hostname}' begins with a numeric subdomain, "
            f"a pattern sometimes used in auto-generated phishing infrastructure.")

    # ── Risk classification ───────────────────────────────────────────────
    if score <= RISK_LOW_MAX:
        risk = "Low"
    elif score <= RISK_MEDIUM_MAX:
        risk = "Medium"
    else:
        risk = "High"

    result["indicators"] = indicators
    result["score"]      = score
    result["risk_level"] = risk
    result["summary"] = (
        f"Analysed URL: {url}\n"
        f"Risk Level: {risk} (score={score})\n"
        f"Indicators detected: {len(indicators)}"
    )
    return result


def print_url_result(res: dict):
    """Pretty-print URL analysis result."""
    sep = "-" * 60
    print(sep)
    print(f"URL       : {res['url']}")
    print(f"Valid     : {res['valid']}")
    print(f"Score     : {res['score']}")
    print(f"Risk      : {res['risk_level']}")
    if res["indicators"]:
        print("Indicators:")
        for ind in res["indicators"]:
            print(f"  [{ind['weight']:+d}] {ind['name']}")
            print(f"       {textwrap.fill(ind['description'], 70, subsequent_indent='       ')}")
    else:
        print("Indicators: None detected")
    print(f"\n{res['disclaimer']}")
    print(sep)


print("URL analysis module loaded.")

URL analysis module loaded.


---

## 5. Message Analysis Module

In [4]:
def analyze_message(text: str) -> dict:
    """
    Perform heuristic analysis of an SMS or email message.

    Parameters
    ----------
    text : str
        The message body to analyse.

    Returns
    -------
    dict
        {
          'text':        str,
          'indicators':  list[dict],
          'score':       int,
          'risk_level':  str,
          'summary':     str,
          'disclaimer':  str
        }
    """
    result = {
        "text": text,
        "indicators": [],
        "score": 0,
        "risk_level": "Low",
        "summary": "",
        "disclaimer": (
            "DISCLAIMER: This is a heuristic analysis only. Many legitimate "
            "messages share surface patterns with phishing messages. A high "
            "score does not prove fraud; a low score does not guarantee "
            "legitimacy. Use multiple sources of evidence."
        )
    }

    if not text or not isinstance(text, str):
        result["summary"] = "Empty or invalid input."
        return result

    text = text.strip()
    result["text"] = text

    indicators = []
    score = 0

    for indicator_name, pattern in MSG_PATTERNS.items():
        matches = pattern.findall(text)
        if matches:
            w = MSG_WEIGHTS.get(indicator_name, 1)
            unique = list(dict.fromkeys(m if isinstance(m, str) else m[0] for m in matches))
            desc = _msg_indicator_description(indicator_name, unique[:5])
            indicators.append({
                "name":        indicator_name,
                "description": desc,
                "weight":      w,
                "matched":     unique[:5]
            })
            score += w

    # Risk classification
    if score <= RISK_LOW_MAX:
        risk = "Low"
    elif score <= RISK_MEDIUM_MAX:
        risk = "Medium"
    else:
        risk = "High"

    result["indicators"] = indicators
    result["score"]      = score
    result["risk_level"] = risk
    result["summary"] = (
        f"Risk Level : {risk} (score={score})\n"
        f"Indicators : {len(indicators)}"
    )
    return result


def _msg_indicator_description(name: str, matched: list) -> str:
    """Return a human-readable description for a message indicator."""
    descs = {
        "urgency_language":       "Creates artificial time pressure to prevent rational evaluation.",
        "prize_or_reward":        "Promises unexpected rewards to lure the recipient.",
        "threat_language":        "Threatens negative consequences to coerce compliance.",
        "sensitive_data_request": "Requests sensitive credentials or personal identifiers.",
        "impersonation":          "Mentions a trusted brand or institution.",
        "suspicious_link":        "Contains a link or call to visit a URL.",
        "generic_greeting":       "Uses a generic greeting instead of the recipient's name.",
        "grammar_errors":         "Contains informal or grammatically unusual language.",
        "call_to_action":         "Uses action verbs pushing the recipient to do something immediately.",
        "otp_or_pin_mention":     "Mentions OTP or PIN — legitimate services never ask for these.",
    }
    base = descs.get(name, "Suspicious pattern detected.")
    sample = ", ".join(f'"{m}"' for m in matched)
    return f"{base} Sample matches: {sample}."


def print_message_result(res: dict):
    """Pretty-print message analysis result."""
    sep = "-" * 60
    print(sep)
    preview = (res["text"][:80] + "...") if len(res["text"]) > 80 else res["text"]
    print(f"Message   : {preview}")
    print(f"Score     : {res['score']}")
    print(f"Risk      : {res['risk_level']}")
    if res["indicators"]:
        print("Indicators:")
        for ind in res["indicators"]:
            print(f"  [{ind['weight']:+d}] {ind['name']}")
            print(f"       {textwrap.fill(ind['description'], 70, subsequent_indent='       ')}")
    else:
        print("Indicators: None detected")
    print(f"\n{res['disclaimer']}")
    print(sep)


print("Message analysis module loaded.")

Message analysis module loaded.


---

## 6. Risk Scoring Module

In [5]:
def classify_risk(score: int) -> str:
    """
    Classify a numeric score into a risk level.

    Scoring rules
    -------------
    0 – 2  : Low     (few or no suspicious signals)
    3 – 5  : Medium  (moderate indicators; warrants caution)
    6+     : High    (multiple strong indicators; treat with suspicion)
    """
    if score <= RISK_LOW_MAX:
        return "Low"
    elif score <= RISK_MEDIUM_MAX:
        return "Medium"
    return "High"


def risk_description(level: str) -> str:
    """Return a human-readable risk-level description."""
    return {
        "Low":    "Few or no suspicious signals detected. Exercise normal caution.",
        "Medium": "Moderate indicators detected. Verify through official channels before acting.",
        "High":   "Multiple strong indicators detected. Treat with significant suspicion and do not act without verification.",
    }.get(level, "Unknown risk level.")


print("Risk scoring module loaded.")

Risk scoring module loaded.


---

## 7. Digital Evidence Integrity Module

In [6]:
def compute_sha256(file_path: str) -> dict:
    """
    Compute the SHA-256 hash of a local file.

    Parameters
    ----------
    file_path : str
        Absolute or relative path to the file.

    Returns
    -------
    dict
        {
          'file_path':  str,
          'sha256':     str | None,
          'file_size':  int | None,
          'timestamp':  str,
          'success':    bool,
          'message':    str,
          'disclaimer': str
        }
    """
    disclaimer = (
        "DISCLAIMER: A matching SHA-256 hash confirms that the file content "
        "has not changed between two measurements. It does NOT verify the "
        "file's authenticity, original creation time, or chain of custody. "
        "For legal proceedings, engage a certified digital forensics examiner."
    )
    ts = datetime.datetime.utcnow().isoformat() + "Z"
    result = {
        "file_path":  file_path,
        "sha256":     None,
        "file_size":  None,
        "timestamp":  ts,
        "success":    False,
        "message":    "",
        "disclaimer": disclaimer
    }

    p = Path(file_path)
    if not p.exists():
        result["message"] = f"File not found: {file_path}"
        return result
    if not p.is_file():
        result["message"] = f"Path is not a file: {file_path}"
        return result

    try:
        h = hashlib.sha256()
        with open(p, "rb") as fh:
            for chunk in iter(lambda: fh.read(65536), b""):
                h.update(chunk)
        result["sha256"]    = h.hexdigest()
        result["file_size"] = p.stat().st_size
        result["success"]   = True
        result["message"]   = "SHA-256 hash computed successfully."
    except PermissionError:
        result["message"] = f"Permission denied reading: {file_path}"
    except OSError as exc:
        result["message"] = f"OS error reading file: {exc}"

    return result


def compute_sha256_from_bytes(data: bytes) -> str:
    """Compute SHA-256 of in-memory bytes (used for synthetic tests)."""
    return hashlib.sha256(data).hexdigest()


def verify_integrity(file_path: str, expected_hash: str) -> dict:
    """
    Verify that a file's SHA-256 hash matches an expected value.

    Returns
    -------
    dict with 'match' (bool), 'computed', 'expected', 'message'
    """
    computed_result = compute_sha256(file_path)
    if not computed_result["success"]:
        return {
            "match":    False,
            "computed": None,
            "expected": expected_hash,
            "message":  computed_result["message"]
        }
    computed = computed_result["sha256"]
    match    = computed.lower() == expected_hash.lower()
    return {
        "match":    match,
        "computed": computed,
        "expected": expected_hash,
        "message":  "Hash matches — file content unchanged." if match
                    else "Hash MISMATCH — file may have been modified."
    }


def print_hash_result(res: dict):
    """Pretty-print SHA-256 hash result."""
    sep = "-" * 60
    print(sep)
    print(f"File      : {res['file_path']}")
    print(f"Success   : {res['success']}")
    if res["success"]:
        print(f"SHA-256   : {res['sha256']}")
        print(f"Size      : {res['file_size']} bytes")
        print(f"Timestamp : {res['timestamp']}")
    print(f"Message   : {res['message']}")
    print(f"\n{res['disclaimer']}")
    print(sep)


print("Digital evidence integrity module loaded.")

Digital evidence integrity module loaded.


---

## 8. Incident Report Generation

In [7]:
def build_incident_record(
    incident_id: str,
    analyst: str,
    description: str,
    url_results: list = None,
    message_results: list = None,
    hash_results: list = None,
    notes: str = ""
) -> dict:
    """
    Assemble a structured incident record from analysis results.

    NOTE: Do NOT include passwords, OTPs, or real personal data in
    the description or notes fields.
    """
    ts = datetime.datetime.utcnow().isoformat() + "Z"

    # Determine overall risk level
    all_risks = []
    if url_results:
        all_risks.extend(r.get("risk_level", "Low") for r in url_results)
    if message_results:
        all_risks.extend(r.get("risk_level", "Low") for r in message_results)

    risk_order = {"High": 3, "Medium": 2, "Low": 1}
    overall_risk = max(all_risks, key=lambda x: risk_order.get(x, 0)) if all_risks else "Low"

    # Build recommendations
    recs = [
        "Do not click, forward, or reply to any suspicious URLs or messages.",
        "Report phishing attempts to your IT / security team.",
        "If credentials may have been exposed, change them immediately via the official service website.",
        "Preserve evidence (screenshots, message headers) without modification.",
    ]
    if overall_risk == "High":
        recs.append("Consider reporting to CERT-In (https://www.cert-in.org.in/) or your national CSIRT.")
        recs.append("Block the sender/domain at the network or email gateway level.")

    record = {
        "incident_id":       incident_id,
        "analyst":           analyst,
        "timestamp":         ts,
        "description":       description,
        "overall_risk":      overall_risk,
        "url_findings":      url_results or [],
        "message_findings":  message_results or [],
        "evidence_hashes":   hash_results or [],
        "recommendations":   recs,
        "notes":             notes,
        "privacy_notice":    "This record contains no passwords, PINs, or personal payment information."
    }
    return record


def export_report_json(record: dict, output_path: str = None) -> str:
    """
    Export an incident record as a formatted JSON string.
    If output_path is provided, write to file as well.
    """
    json_str = json.dumps(record, indent=2, default=str)
    if output_path:
        with open(output_path, "w", encoding="utf-8") as fh:
            fh.write(json_str)
        print(f"Report saved to: {output_path}")
    return json_str


def export_report_text(record: dict, output_path: str = None) -> str:
    """
    Export an incident record as a plain-text report.
    """
    lines = []
    lines.append("=" * 70)
    lines.append("           SENTINELX INCIDENT REPORT")
    lines.append("=" * 70)
    lines.append(f"Incident ID   : {record['incident_id']}")
    lines.append(f"Analyst       : {record['analyst']}")
    lines.append(f"Timestamp     : {record['timestamp']}")
    lines.append(f"Overall Risk  : {record['overall_risk']}")
    lines.append(f"Description   : {record['description']}")
    lines.append("")

    # URL findings
    if record["url_findings"]:
        lines.append("-" * 50)
        lines.append("URL FINDINGS")
        lines.append("-" * 50)
        for i, uf in enumerate(record["url_findings"], 1):
            lines.append(f"  [{i}] URL: {uf.get('url', 'N/A')}")
            lines.append(f"      Risk: {uf.get('risk_level', 'N/A')} (score={uf.get('score', 0)})")
            for ind in uf.get("indicators", []):
                lines.append(f"        • {ind['name']} (+{ind['weight']})")
        lines.append("")

    # Message findings
    if record["message_findings"]:
        lines.append("-" * 50)
        lines.append("MESSAGE FINDINGS")
        lines.append("-" * 50)
        for i, mf in enumerate(record["message_findings"], 1):
            preview = (mf.get("text", "")[:60] + "...") if len(mf.get("text", "")) > 60 else mf.get("text", "")
            lines.append(f"  [{i}] Message: {preview}")
            lines.append(f"      Risk: {mf.get('risk_level', 'N/A')} (score={mf.get('score', 0)})")
            for ind in mf.get("indicators", []):
                lines.append(f"        • {ind['name']} (+{ind['weight']})")
        lines.append("")

    # Evidence hashes
    if record["evidence_hashes"]:
        lines.append("-" * 50)
        lines.append("EVIDENCE INTEGRITY")
        lines.append("-" * 50)
        for hr in record["evidence_hashes"]:
            lines.append(f"  File: {hr.get('file_path', 'N/A')}")
            lines.append(f"  SHA-256: {hr.get('sha256', 'N/A')}")
            lines.append(f"  Timestamp: {hr.get('timestamp', 'N/A')}")
        lines.append("")

    # Recommendations
    lines.append("-" * 50)
    lines.append("RECOMMENDATIONS")
    lines.append("-" * 50)
    for rec in record["recommendations"]:
        lines.append(f"  • {rec}")
    lines.append("")

    if record.get("notes"):
        lines.append(f"Notes: {record['notes']}")
        lines.append("")

    lines.append(record["privacy_notice"])
    lines.append("=" * 70)

    report_text = "\n".join(lines)
    if output_path:
        with open(output_path, "w", encoding="utf-8") as fh:
            fh.write(report_text)
        print(f"Text report saved to: {output_path}")
    return report_text


print("Incident report module loaded.")

Incident report module loaded.


---

## 9. Synthetic Demonstration Data

In [8]:
# ── Synthetic URLs for demonstration ─────────────────────────────────────
# All URLs below are SYNTHETIC. They are not real URLs and are not visited.

DEMO_URLS = [
    # Legitimate-looking
    "https://www.github.com/features",
    "https://en.wikipedia.org/wiki/Phishing",
    # Suspicious: IP address host
    "http://192.168.1.1/login/verify?user=admin",
    # Suspicious: URL shortener
    "https://bit.ly/3xZ9fakelink",
    # Suspicious: misleading brand + suspicious TLD
    "http://sbi-bank-secure-login.xyz/account/verify",
    # Suspicious: @ in URL
    "http://legit.bank.com@evil-phish.tk/login",
    # Suspicious: excessive subdomains + suspicious keywords
    "http://secure.login.verify.update.example.tk/account/password",
    # HTTP only with login path
    "http://mybank.com/signin",
]

# ── Synthetic messages for demonstration ─────────────────────────────────
DEMO_MESSAGES = [
    # Legitimate-looking
    "Hi Manvith, your Amazon order #104-2345678 has been dispatched and will arrive tomorrow.",
    "Reminder: your electricity bill payment of ₹850 is due on 30th June.",
    # Phishing-like
    ("Dear Customer, your SBI account has been SUSPENDED due to suspicious activity. "
     "Verify your OTP immediately at http://sbi-verify-now.tk or your account will be terminated."),
    ("CONGRATULATIONS! You have WON a prize of ₹5,00,000 in the lucky draw. "
     "Claim your reward now. Reply with your bank account number and PAN number."),
    ("URGENT: Your HDFC credit card is blocked. Call 1800-FAKE-NUM or click "
     "https://bit.ly/hdfc-unblock to update your CVV and PIN immediately."),
    # Low risk / ambiguous
    "Your OTP for logging into ExampleApp is 483921. Valid for 5 minutes. Do not share.",
]

print("Synthetic demonstration data ready.")

Synthetic demonstration data ready.


---

## 10. Feature Demonstration

In [9]:
print("\n" + "=" * 60)
print(" DEMO: URL ANALYSIS")
print("=" * 60)

url_demo_results = []
for demo_url in DEMO_URLS:
    res = analyze_url(demo_url)
    url_demo_results.append(res)
    print_url_result(res)


 DEMO: URL ANALYSIS
------------------------------------------------------------
URL       : https://www.github.com/features
Valid     : True
Score     : 0
Risk      : Low
Indicators: None detected

DISCLAIMER: This is a heuristic analysis only. A high score does not confirm that the URL is malicious. A low score does not confirm safety. Never visit unrecognised URLs.
------------------------------------------------------------
------------------------------------------------------------
URL       : https://en.wikipedia.org/wiki/Phishing
Valid     : True
Score     : 0
Risk      : Low
Indicators: None detected

DISCLAIMER: This is a heuristic analysis only. A high score does not confirm that the URL is malicious. A low score does not confirm safety. Never visit unrecognised URLs.
------------------------------------------------------------
------------------------------------------------------------
URL       : http://192.168.1.1/login/verify?user=admin
Valid     : True
Score     : 9
R

In [10]:
print("\n" + "=" * 60)
print(" DEMO: MESSAGE ANALYSIS")
print("=" * 60)

msg_demo_results = []
for demo_msg in DEMO_MESSAGES:
    res = analyze_message(demo_msg)
    msg_demo_results.append(res)
    print_message_result(res)


 DEMO: MESSAGE ANALYSIS
------------------------------------------------------------
Message   : Hi Manvith, your Amazon order #104-2345678 has been dispatched and will arrive t...
Score     : 2
Risk      : Low
Indicators:
  [+2] impersonation
       Mentions a trusted brand or institution. Sample matches: "Amazon".

DISCLAIMER: This is a heuristic analysis only. Many legitimate messages share surface patterns with phishing messages. A high score does not prove fraud; a low score does not guarantee legitimacy. Use multiple sources of evidence.
------------------------------------------------------------
------------------------------------------------------------
Message   : Reminder: your electricity bill payment of ₹850 is due on 30th June.
Score     : 0
Risk      : Low
Indicators: None detected

DISCLAIMER: This is a heuristic analysis only. Many legitimate messages share surface patterns with phishing messages. A high score does not prove fraud; a low score does not guarantee legi

In [11]:
# ── Demonstrate evidence integrity using a synthetic file ────────────────
print("\n" + "=" * 60)
print(" DEMO: DIGITAL EVIDENCE INTEGRITY")
print("=" * 60)

# Create a synthetic evidence file
EVIDENCE_FILE = "synthetic_evidence.txt"
evidence_content = (
    "SYNTHETIC EVIDENCE FILE\n"
    "Created for SentinelX demonstration purposes only.\n"
    "This file is NOT real evidence.\n"
    "Timestamp: " + datetime.datetime.utcnow().isoformat() + "\n"
)
with open(EVIDENCE_FILE, "w", encoding="utf-8") as f:
    f.write(evidence_content)
print(f"Synthetic evidence file created: {EVIDENCE_FILE}")

# First hash computation
hash_result_1 = compute_sha256(EVIDENCE_FILE)
print("\n--- First hash computation ---")
print_hash_result(hash_result_1)

# Second hash computation (no change → hashes must match)
hash_result_2 = compute_sha256(EVIDENCE_FILE)
print("\n--- Second hash computation (file unchanged) ---")
match = hash_result_1["sha256"] == hash_result_2["sha256"]
print(f"Hash 1: {hash_result_1['sha256']}")
print(f"Hash 2: {hash_result_2['sha256']}")
print(f"Match : {match} — {'File content verified consistent.' if match else 'MISMATCH detected!'}")

# Simulate modification and re-hash
with open(EVIDENCE_FILE, "a", encoding="utf-8") as f:
    f.write("[MODIFIED]\n")
hash_result_3 = compute_sha256(EVIDENCE_FILE)
print("\n--- Hash after file modification ---")
match2 = hash_result_1["sha256"] == hash_result_3["sha256"]
print(f"Original Hash : {hash_result_1['sha256']}")
print(f"Modified Hash : {hash_result_3['sha256']}")
print(f"Match         : {match2} — {'MISMATCH — modification detected!' if not match2 else 'Unexpected match.'}")


 DEMO: DIGITAL EVIDENCE INTEGRITY
Synthetic evidence file created: synthetic_evidence.txt

--- First hash computation ---
------------------------------------------------------------
File      : synthetic_evidence.txt
Success   : True
SHA-256   : 777024468b1b3c6e2458e7acadbec478cda0d1076a53e70278a7d02abccd718d
Size      : 149 bytes
Timestamp : 2026-10-22T12:20:11.065517Z
Message   : SHA-256 hash computed successfully.

DISCLAIMER: A matching SHA-256 hash confirms that the file content has not changed between two measurements. It does NOT verify the file's authenticity, original creation time, or chain of custody. For legal proceedings, engage a certified digital forensics examiner.
------------------------------------------------------------

--- Second hash computation (file unchanged) ---
Hash 1: 777024468b1b3c6e2458e7acadbec478cda0d1076a53e70278a7d02abccd718d
Hash 2: 777024468b1b3c6e2458e7acadbec478cda0d1076a53e70278a7d02abccd718d
Match : True — File content verified consistent.

-

In [12]:
# ── Build and print an incident report ───────────────────────────────────
print("\n" + "=" * 60)
print(" DEMO: INCIDENT REPORT GENERATION")
print("=" * 60)

incident = build_incident_record(
    incident_id      = "INC-2026-001",
    analyst          = "Manvith (Trainee Analyst, IBM SkillsBuild 2026)",
    description      = (
        "User reported receiving a suspicious SMS claiming to be from SBI bank, "
        "requesting OTP verification via an unfamiliar URL. "
        "Synthetic example used for demonstration."
    ),
    url_results      = [url_demo_results[4], url_demo_results[5]],  # suspicious ones
    message_results  = [msg_demo_results[2], msg_demo_results[3]],  # phishing-like
    hash_results     = [hash_result_1],
    notes            = "All data is synthetic. No real personal information included."
)

text_report = export_report_text(incident, output_path="sentinelx_report.txt")
json_report = export_report_json(incident, output_path="sentinelx_report.json")

print("\n" + text_report)


 DEMO: INCIDENT REPORT GENERATION
Text report saved to: sentinelx_report.txt
Report saved to: sentinelx_report.json

           SENTINELX INCIDENT REPORT
Incident ID   : INC-2026-001
Analyst       : Manvith (Trainee Analyst, IBM SkillsBuild 2026)
Timestamp     : 2026-10-22T12:20:11.110760Z
Overall Risk  : High
Description   : User reported receiving a suspicious SMS claiming to be from SBI bank, requesting OTP verification via an unfamiliar URL. Synthetic example used for demonstration.

--------------------------------------------------
URL FINDINGS
--------------------------------------------------
  [1] URL: http://sbi-bank-secure-login.xyz/account/verify
      Risk: Medium (score=5)
        • suspicious_tld (+2)
        • suspicious_keywords (+2)
        • http_not_https (+1)
  [2] URL: http://legit.bank.com@evil-phish.tk/login
      Risk: High (score=8)
        • suspicious_tld (+2)
        • suspicious_keywords (+2)
        • at_symbol_in_url (+3)
        • http_not_https (+1)



---

## 11. Test Suite

The following cells define and immediately execute at least **10 meaningful test cases**. Each test prints its expected result, actual result, and PASS/FAIL status.

In [13]:
# ── Test framework ─────────────────────────────────────────────────────
class TestRunner:
    def __init__(self):
        self.passed = 0
        self.failed = 0
        self.results = []

    def run(self, name, expected, actual, cmp=None):
        """Run a single assertion."""
        if cmp is None:
            ok = expected == actual
        else:
            ok = cmp(expected, actual)
        status = "PASS" if ok else "FAIL"
        if ok:
            self.passed += 1
        else:
            self.failed += 1
        self.results.append({
            "name":     name,
            "expected": expected,
            "actual":   actual,
            "status":   status
        })
        mark = "PASS" if ok else "FAIL"
        print(f"  [{mark}]  {name}")
        if not ok:
            print(f"         Expected : {expected!r}")
            print(f"         Actual   : {actual!r}")
        return ok

    def summary(self):
        total = self.passed + self.failed
        print("\n" + "=" * 60)
        print(f" TEST SUMMARY: {self.passed}/{total} passed, {self.failed}/{total} failed")
        print("=" * 60)
        return self.failed == 0


runner = TestRunner()
print("Test framework ready.")

Test framework ready.


In [14]:
print("\n" + "=" * 60)
print(" RUNNING TEST CASES")
print("=" * 60)

# ── TC-01: Legitimate HTTPS URL → Low risk ────────────────────────────
res = analyze_url("https://www.github.com/features")
runner.run(
    "TC-01: Legitimate HTTPS URL → Low risk",
    "Low", res["risk_level"]
)

# ── TC-02: IP address host → High risk ───────────────────────────────
res = analyze_url("http://192.168.10.5/login?user=admin")
runner.run(
    "TC-02: IP address in URL host → High risk",
    "High", res["risk_level"]
)

# ── TC-03: URL shortener detected ─────────────────────────────────────
res = analyze_url("https://bit.ly/3xZ9fake")
detected = any(i["name"] == "url_shortener" for i in res["indicators"])
runner.run(
    "TC-03: URL shortener (bit.ly) indicator detected",
    True, detected
)

# ── TC-04: Suspicious TLD (.tk) → Medium or High ─────────────────────
res = analyze_url("http://mysite.tk/verify")
runner.run(
    "TC-04: Suspicious TLD .tk → Medium or High",
    True, res["risk_level"] in ("Medium", "High")
)

# ── TC-05: @ symbol in URL → at_symbol_in_url indicator ──────────────
res = analyze_url("http://legit.bank.com@evil.tk/login")
detected = any(i["name"] == "at_symbol_in_url" for i in res["indicators"])
runner.run(
    "TC-05: @ symbol in URL → indicator detected",
    True, detected
)

# ── TC-06: Empty URL input → invalid ─────────────────────────────────
res = analyze_url("")
runner.run(
    "TC-06: Empty URL → valid=False",
    False, res["valid"]
)

# ── TC-07: Malformed URL (no hostname) → invalid ──────────────────────
res = analyze_url("http://")
runner.run(
    "TC-07: Malformed URL with no hostname → valid=False",
    False, res["valid"]
)

# ── TC-08: Phishing SMS → High risk ──────────────────────────────────
phishing_sms = (
    "Dear Customer, your SBI account is SUSPENDED. Verify your OTP now "
    "at http://sbi-fake-login.tk or face legal action and account termination."
)
res = analyze_message(phishing_sms)
runner.run(
    "TC-08: Classic phishing SMS → High risk",
    "High", res["risk_level"]
)

# ── TC-09: Ordinary message → Low risk ───────────────────────────────
normal_msg = "Team lunch is at 1 PM in the cafeteria. See you there!"
res = analyze_message(normal_msg)
runner.run(
    "TC-09: Ordinary message → Low risk",
    "Low", res["risk_level"]
)

# ── TC-10: Empty message → Low risk / empty indicators ────────────────
res = analyze_message("")
runner.run(
    "TC-10: Empty message → score 0, 0 indicators",
    0, res["score"]
)

# ── TC-11: SHA-256 consistency (in-memory bytes) ─────────────────────
data = b"SentinelX test data 2026"
h1 = compute_sha256_from_bytes(data)
h2 = compute_sha256_from_bytes(data)
runner.run(
    "TC-11: SHA-256 of identical bytes → same hash",
    True, h1 == h2
)

# ── TC-12: Modified bytes → different SHA-256 ────────────────────────
data_modified = data + b"X"
h3 = compute_sha256_from_bytes(data_modified)
runner.run(
    "TC-12: Modified bytes → different SHA-256",
    True, h1 != h3
)

# ── TC-13: File-based hash consistency ───────────────────────────────
test_file = "_sentinelx_test_tmp.txt"
with open(test_file, "w") as f:
    f.write("Test content for SentinelX hash verification.")
hr1 = compute_sha256(test_file)
hr2 = compute_sha256(test_file)
runner.run(
    "TC-13: File SHA-256 — same file → same hash",
    True, hr1["sha256"] == hr2["sha256"]
)
os.remove(test_file)  # clean up

# ── TC-14: Non-existent file → success=False ─────────────────────────
hr_missing = compute_sha256("does_not_exist_xyz.bin")
runner.run(
    "TC-14: Non-existent file → success=False",
    False, hr_missing["success"]
)

# ── TC-15: Risk classify boundary ─────────────────────────────────────
runner.run("TC-15a: classify_risk(0) → Low",    "Low",    classify_risk(0))
runner.run("TC-15b: classify_risk(2) → Low",    "Low",    classify_risk(2))
runner.run("TC-15c: classify_risk(3) → Medium", "Medium", classify_risk(3))
runner.run("TC-15d: classify_risk(5) → Medium", "Medium", classify_risk(5))
runner.run("TC-15e: classify_risk(6) → High",   "High",   classify_risk(6))
runner.run("TC-15f: classify_risk(99) → High",  "High",   classify_risk(99))

# ── TC-16: Brand impersonation in URL (paypal as subdomain of different domain) ──
res = analyze_url("http://paypal.secure-evil-site.tk/account/verify")
detected = any(i["name"] == "misleading_brand" for i in res["indicators"])
runner.run(
    "TC-16: Brand 'paypal' in subdomain of different domain → misleading_brand",
    True, detected
)

# ── TC-17: Incident record overall risk elevation ─────────────────────
high_url = analyze_url("http://192.168.1.1/login?verify=otp")
record = build_incident_record(
    "INC-TEST-001", "AutoTest", "Test incident",
    url_results=[high_url]
)
runner.run(
    "TC-17: Incident with High-risk URL → overall_risk=High",
    "High", record["overall_risk"]
)

# ── TC-18: JSON report is valid JSON ──────────────────────────────────
low_url = analyze_url("https://www.example.com")
sample_record = build_incident_record("INC-TEST-002", "AutoTest", "JSON validity test",
                                      url_results=[low_url])
json_str = export_report_json(sample_record)
try:
    parsed_back = json.loads(json_str)
    json_valid = isinstance(parsed_back, dict)
except Exception:
    json_valid = False
runner.run(
    "TC-18: Exported JSON report is valid, parseable JSON",
    True, json_valid
)

# ── TC-19: OTP in message indicator ──────────────────────────────────
otp_msg = "Your OTP is 123456. Do not share your OTP with anyone."
res = analyze_message(otp_msg)
detected = any(i["name"] == "otp_or_pin_mention" for i in res["indicators"])
runner.run(
    "TC-19: 'OTP' in message → otp_or_pin_mention indicator",
    True, detected
)

# ── TC-20: Verify integrity helper detects tampering ─────────────────
tamper_file = "_sentinelx_tamper_test.txt"
with open(tamper_file, "w") as f:
    f.write("Original content.")
original_hash = compute_sha256(tamper_file)["sha256"]
# Tamper
with open(tamper_file, "a") as f:
    f.write(" Tampered!")
vi_result = verify_integrity(tamper_file, original_hash)
os.remove(tamper_file)  # clean up
runner.run(
    "TC-20: verify_integrity detects tampering → match=False",
    False, vi_result["match"]
)

# ── Print summary ──────────────────────────────────────────────────────
all_passed = runner.summary()


 RUNNING TEST CASES
  [PASS]  TC-01: Legitimate HTTPS URL → Low risk
  [PASS]  TC-02: IP address in URL host → High risk
  [PASS]  TC-03: URL shortener (bit.ly) indicator detected
  [PASS]  TC-04: Suspicious TLD .tk → Medium or High
  [PASS]  TC-05: @ symbol in URL → indicator detected
  [PASS]  TC-06: Empty URL → valid=False
  [PASS]  TC-07: Malformed URL with no hostname → valid=False
  [PASS]  TC-08: Classic phishing SMS → High risk
  [PASS]  TC-09: Ordinary message → Low risk
  [PASS]  TC-10: Empty message → score 0, 0 indicators
  [PASS]  TC-11: SHA-256 of identical bytes → same hash
  [PASS]  TC-12: Modified bytes → different SHA-256
  [PASS]  TC-13: File SHA-256 — same file → same hash
  [PASS]  TC-14: Non-existent file → success=False
  [PASS]  TC-15a: classify_risk(0) → Low
  [PASS]  TC-15b: classify_risk(2) → Low
  [PASS]  TC-15c: classify_risk(3) → Medium
  [PASS]  TC-15d: classify_risk(5) → Medium
  [PASS]  TC-15e: classify_risk(6) → High
  [PASS]  TC-15f: classify_risk(99

---

## 12. Limitations

1. **Heuristic-only analysis**: SentinelX uses pattern matching and structural checks. It does not query live threat-intelligence feeds (e.g., VirusTotal, Google Safe Browsing). A URL that passes all heuristic checks may still be malicious.

2. **No URL fetching**: The prototype never fetches or renders a URL. It cannot detect phishing content that lives only on the rendered page (JavaScript-driven, image-based, or redirected pages).

3. **False positives and false negatives**: Legitimate organisations sometimes use URL shorteners, non-standard ports, or urgency language. Conversely, sophisticated phishing pages can pass all structural checks.

4. **Language coverage**: Message analysis patterns are primarily in English and some Indian-language transliterations. Multi-lingual phishing (e.g., Hindi, Tamil, Telugu scripts) is not fully covered.

5. **Evidence integrity scope**: SHA-256 hashing proves content consistency between two measurements. It does not establish authenticity, provenance, chain-of-custody, or timestamp accuracy.

6. **No real-time data**: The suspicious-TLD and brand-keyword lists are static and may become outdated as threat actors register new domains.

7. **Prototype status**: This is an educational prototype. It should not be used as the sole decision-making tool for legal or security-enforcement purposes.

---

## 13. Conclusion

SentinelX demonstrates that meaningful cyber-fraud awareness tooling can be built entirely in the Python standard library — without paid APIs, external databases, or server infrastructure. The platform covers four key areas relevant to digital safety and forensics:

* **URL analysis** — structural heuristics to flag common phishing URL patterns.
* **Message analysis** — keyword and pattern matching against known phishing language.
* **Risk classification** — a transparent scoring model with documented thresholds.
* **Evidence integrity** — SHA-256 hashing to detect file tampering.

All 20 test cases executed successfully in the current environment. The project is fully self-contained, privacy-respecting, and appropriate for educational and awareness-raising purposes within the IBM SkillsBuild Cyber Security BootCamp 2026.

---
*End of SentinelX Notebook*